# Lab 5 보충 — 최종 모델을 15만 건 전체로 학습해 저장

노트북 05에서는 Lab 5 코드대로 67%로 학습하고 33%로 '저장 → 다시 불러오기'를 확인했다.
실제로 쓸 모델은 가진 데이터를 모두 써서 학습하는 것이 맞으므로, 같은 설정(데이터 정리 + RF 잎 최소 176명)을 **15만 건 전체**로 다시 학습해 저장한다.

- 전체로 학습하면 남는 시험 데이터가 없다. 그래서 이 모델의 성능 추정치는 **노트북 05의 층화 10-fold AUC 0.856**을 그대로 쓴다
  (10-fold는 매번 13.5만 건으로 학습했으니 15만 건 모델은 그와 같거나 조금 더 좋다).

In [1]:
import os, sys, time
import numpy, sklearn, joblib
from pandas import read_csv, DataFrame
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

names = ['target', 'revol', 'age', 'late30', 'debt', 'income', 'depend', 'defaulted', 'loans']
features = names[1:]
dataframe = read_csv('../data/givemesomecredit.csv', names=names, header=0)
X = dataframe.values[:, 1:9].astype(float)
Y = dataframe.values[:, 0].astype(int)

def clean(A):
    """노트북 05와 같은 정리 단계: late30의 96/98 → 0 + 표시 열, 소득 대체값(6638/6746) 표시 열."""
    A = A.copy()
    code = numpy.isin(A[:, 2], [96, 98]).astype(float)
    fill = numpy.isin(A[:, 4], [6638, 6746]).astype(float)
    A[code == 1, 2] = 0
    return numpy.column_stack([A, code, fill])

final = Pipeline([('clean', FunctionTransformer(clean)),
                  ('rf', RandomForestClassifier(n_estimators=100, max_features=3, min_samples_leaf=176, random_state=7, n_jobs=-1))])
t0 = time.time()
final.fit(X, Y)
print(f'15만 건 전체 학습: {time.time() - t0:.1f}초')

15만 건 전체 학습: 3.1초


In [2]:
joblib.dump(final, 'finalized_model_full.sav')
loaded = joblib.load('finalized_model_full.sav')
p1 = final.predict_proba(X)[:, 1]
p2 = loaded.predict_proba(X)[:, 1]
print(f'finalized_model_full.sav: {os.path.getsize("finalized_model_full.sav") / 1e6:.1f} MB')
print('다시 불러온 모델의 예측이 저장 전과 같은가:', numpy.allclose(p1, p2))
print(f'학습 데이터 AUC (참고용, 실제보다 높게 나옴): {roc_auc_score(Y, p1):.4f}  ← 성능 추정은 10-fold 0.856을 사용')
print('Python', sys.version.split()[0], '· scikit-learn', sklearn.__version__, '· numpy', numpy.__version__)

finalized_model_full.sav: 5.5 MB
다시 불러온 모델의 예측이 저장 전과 같은가: True
학습 데이터 AUC (참고용, 실제보다 높게 나옴): 0.8734  ← 성능 추정은 10-fold 0.856을 사용
Python 3.13.2 · scikit-learn 1.9.1 · numpy 2.5.4


## 최종 모델이 중요하게 본 변수 (트리 불순도 감소량)

In [3]:
imp = DataFrame({'중요도': final.named_steps['rf'].feature_importances_},
                index=features + ['late30_코드표시', '소득_대체값표시']).sort_values('중요도', ascending=False)
print(imp.round(3))

               중요도
defaulted    0.388
revol        0.342
late30       0.152
age          0.035
debt         0.031
income       0.028
loans        0.014
depend       0.006
late30_코드표시  0.003
소득_대체값표시     0.002


## 위험 점수 상위 고객에 연체자가 얼마나 몰리나 (학습 데이터 기준 · 참고용)

In [4]:
order = numpy.argsort(-p1)
for top in [0.05, 0.10, 0.20]:
    k = int(len(Y) * top)
    caught = Y[order[:k]].sum()
    print(f'위험 점수 상위 {top:.0%} ({k:,}명): 연체자 {caught:,}명 = 전체 연체자의 {caught / Y.sum():.1%}')

위험 점수 상위 5% (7,500명): 연체자 3,319명 = 전체 연체자의 33.1%
위험 점수 상위 10% (15,000명): 연체자 5,233명 = 전체 연체자의 52.2%
위험 점수 상위 20% (30,000명): 연체자 7,325명 = 전체 연체자의 73.1%


## 같은 질문을 '처음 보는 고객' 기준으로 (층화 10-fold 예측)
각 고객의 점수를 그 고객을 학습에 쓰지 않은 모델로 매긴다 → 실제 성능에 가까운 숫자.

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=7)
t0 = time.time()
oof = cross_val_predict(final, X, Y, cv=kfold, method='predict_proba')[:, 1]
print(f'10-fold 예측 AUC: {roc_auc_score(Y, oof):.4f} · {time.time() - t0:.0f}초')
order = numpy.argsort(-oof)
for top in [0.05, 0.10, 0.20]:
    k = int(len(Y) * top)
    caught = Y[order[:k]].sum()
    print(f'위험 점수 상위 {top:.0%} ({k:,}명): 연체자 {caught:,}명 = 전체 연체자의 {caught / Y.sum():.1%} · 그룹 연체율 {caught / k:.1%}')

10-fold 예측 AUC: 0.8555 · 50초
위험 점수 상위 5% (7,500명): 연체자 3,204명 = 전체 연체자의 32.0% · 그룹 연체율 42.7%
위험 점수 상위 10% (15,000명): 연체자 5,133명 = 전체 연체자의 51.2% · 그룹 연체율 34.2%
위험 점수 상위 20% (30,000명): 연체자 7,128명 = 전체 연체자의 71.1% · 그룹 연체율 23.8%
